<!-- dd:dd-lesson-eo-2 -->

# Reduce

*Einops · `eo-2`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# The side panel decides what you practise next. This cell only tells
# the notebook who you are, for the completion beacon.
DD_TOKEN = ""  # paste from the extension's Settings if you want beacons
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"
DD_LESSON_ID = "eo-2"


<!-- dd:dd-kp-einops-reduce-model -->

## einops.reduce — dropping axes with an aggregation

`einops.reduce-model`


Where rearrange must keep every axis, **`einops.reduce` is allowed to drop
them — and you say HOW the dropped values collapse**:

> `einops.reduce(x, 'b c h w -> b', 'mean')`
> — c, h, w vanish from the pattern, so each output element is the mean
> over everything that vanished. The third argument names the aggregation:
> `'mean'`, `'max'`, `'min'`, `'sum'`, `'prod'`.

This is PyTorch's `dim=` reductions with the einsum-style deletion rule, in
einops clothing — three notations, one concept:

- `x.mean(dim=(1, 2, 3))` — axes by number.
- `t.einsum('bchw->b', x) / (c*h*w)` — sum by deletion, mean by hand.
- `reduce(x, 'b c h w -> b', 'mean')` — deletion by name, aggregation
  declared. (Note: unlike einsum, reduce does means/maxes natively — no
  divide-outside dance.)

Details that matter in the drills:

- **Partial drops**: `'b h w c -> b h w'` maxes only over channels;
  `'b c h w -> b c'` averages each channel map. Any subset of axes can go.
- **Keep a singleton**: writing `1` (or `()` — same thing) in the output
  where the dropped axis was — `'h w c -> 1 w c'` — keeps the result
  broadcast-ready against the input, einops' keepdim. This is exactly the
  `keepdim=True` story: reduce-then-broadcast pipelines (subtract each
  column's max…) want the singleton kept.
- **Reduce + rearrange compose**: the pattern can still permute the
  survivors while reducing (`'b c h w -> c b'` is legal), and — the next
  KP — parenthesized factors turn reduce into pooling.


Task: per-image scalar means; per-column max keeping a singleton row; and a
grayscale via mean-over-channels with the channel kept.


In [ ]:
import torch as t
import einops

x = t.arange(24.0).reshape(2, 3, 2, 2)      # (b, c, h, w)

# All of c, h, w collapse under 'mean' -> one number per image.
per_image = einops.reduce(x, 'b c h w -> b', 'mean')
assert per_image.tolist() == [5.5, 17.5]
assert t.allclose(per_image, x.mean(dim=(1, 2, 3)))   # the numpy twin

# Partial drop: mean over spatial only -> per-channel statistics.
per_channel = einops.reduce(x, 'b c h w -> b c', 'mean')
assert per_channel.shape == (2, 3)
assert per_channel[0, 0] == x[0, 0].mean()

# Keep-a-singleton: per-column max of an image, row axis kept as 1.
img = t.tensor([[1.0, 5.0],
                [7.0, 2.0]])
colmax = einops.reduce(img, 'h w -> 1 w', 'max')
assert colmax.shape == (1, 2)
assert colmax.tolist() == [[7.0, 5.0]]
# Why keep it: the (1, 2) result broadcasts straight back against (2, 2).
assert (img - colmax).shape == (2, 2)

# Grayscale keeping a trailing singleton channel: (b,h,w,c) -> (b,h,w,1).
imgs = t.ones((2, 2, 2, 3))
gray = einops.reduce(imgs, 'b h w c -> b h w 1', 'mean')
assert gray.shape == (2, 2, 2, 1)
print("'b c h w -> b'  ", per_image, " (one number per image)")
print("'b c h w -> b c'", tuple(per_channel.shape), per_channel[0])
print("'h w -> 1 w'    ", colmax, "shape", tuple(colmax.shape),
      "-> broadcasts back to", tuple((img - colmax).shape))
print("grayscale keeps the channel axis:", tuple(imgs.shape), "->",
      tuple(gray.shape))




Why each step:

1. The `dim=` twin assert carries your existing axis intuition into the new
   notation: names deleted ↔ axis numbers listed. After a few reps the
   named form usually reads faster — especially at rank 4+.
2. In the singleton example, the follow-up subtraction is the POINT: the
   kept `1` is what makes reduce-then-operate pipelines shape-safe, same as
   keepdims in np-3.
3. The grayscale line matches a drill's exact contract ('-> b h w 1');
   note reduce handles mean natively — resist the einsum habit of dividing
   afterwards.


In [ ]:
import torch as t
import einops

x = t.arange(24.0).reshape(2, 3, 2, 2)      # (b, c, h, w)

# All of c, h, w collapse under 'mean' -> one number per image.
per_image = einops.reduce(x, 'b c h w -> b', 'mean')
assert per_image.tolist() == [5.5, 17.5]
assert t.allclose(per_image, x.mean(dim=(1, 2, 3)))   # the numpy twin

# Partial drop: mean over spatial only -> per-channel statistics.
per_channel = einops.reduce(x, 'b c h w -> b c', 'mean')
assert per_channel.shape == (2, 3)
assert per_channel[0, 0] == x[0, 0].mean()

# Keep-a-singleton: per-column max of an image, row axis kept as 1.
img = t.tensor([[1.0, 5.0],
                [7.0, 2.0]])
colmax = einops.reduce(img, 'h w -> 1 w', 'max')
assert colmax.shape == (1, 2)
assert colmax.tolist() == [[7.0, 5.0]]
# Why keep it: the (1, 2) result broadcasts straight back against (2, 2).
assert (img - colmax).shape == (2, 2)

# Grayscale keeping a trailing singleton channel: (b,h,w,c) -> (b,h,w,1).
imgs = t.ones((2, 2, 2, 3))
gray = einops.reduce(imgs, 'b h w c -> b h w 1', 'mean')
assert gray.shape == (2, 2, 2, 1)
print("'b c h w -> b'  ", per_image, " (one number per image)")
print("'b c h w -> b c'", tuple(per_channel.shape), per_channel[0])
print("'h w -> 1 w'    ", colmax, "shape", tuple(colmax.shape),
      "-> broadcasts back to", tuple((img - colmax).shape))
print("grayscale keeps the channel axis:", tuple(imgs.shape), "->",
      tuple(gray.shape))


<!-- dd:dd-q325 -->

### Problem 325 · faded

Each image of a channels-first batch → one scalar mean.


In [ ]:
import torch as t
import einops

def solve(arr):
    """(b, c, h, w) -> (b,): mean over channels and pixels."""
    return einops.reduce(arr, '_____', 'mean')


<!-- dd:dd-q328 -->

### Problem 328 · guided

Write a function solve(img) that takes a channels-last image of shape (h, w, c) and returns the (w, c) tensor averaging over the HEIGHT axis: each (column, channel) pair's mean down the image.


<details>
<summary>Hints</summary>

1. (h, w, c) image, average over the HEIGHT axis only → (w, c).
2. One name disappears from the pattern; the aggregation string says how.
3. `einops.reduce(img, 'h w c -> w c', 'mean')`.

</details>


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    """Return the (w, c) array averaging over the HEIGHT axis."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 2, 2)))


<!-- dd:dd-q326 -->

### Problem 326 · independent

Write a function solve(arr) that takes a channels-last batch of shape (b, h, w, c) and returns the (b, h, w) tensor where each output pixel is the MAXIMUM across that pixel's channels.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return the (b, h, w) array where each output pixel is the MAXIMUM across that pixel's channels."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


<!-- dd:dd-q367 -->

### Problem 367 · independent

Write a function solve(arr) that takes a channels-first batch (b, c, h, w) and returns the (b, c) tensor of PER-CHANNEL spatial means: each value averages one channel's h x w map.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return the (b, c) array of PER-CHANNEL spatial means."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


<!-- dd:dd-q399 -->

### Problem 399 · independent

Write solve(imgs) for a float (B, H, W, C) batch: grayscale each image by AVERAGING across the color channels, keeping a trailing singleton channel — output (B, H, W, 1). Pattern: einops.reduce 'b h w c -> b h w ()' with 'mean'.


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')).to(t.float32), 1, -1)
imgs = arr

def solve(imgs):
    # Write your solution here
    return None

print(solve(imgs))


<!-- dd:dd-q402 -->

### Problem 402 · independent

Write solve(img) for an (H, W, C) image: compute each COLUMN's per-channel maximum (reduce the height axis to a singleton: 'h w c -> () w c' with 'max') and subtract it from the image. Return img - mx. (uint8 wraparound on underflow is expected — same on both sides.)


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img = arr[3]

def solve(img):
    # Write your solution here
    return None

print(solve(img))


<!-- dd:dd-q332 -->

### Problem 332 · independent

Write solve(img) for an (H, W, C) image: compute the maximum over each ROW — reducing width and channels to singletons, 'h w c -> h () c' is close but the row max is over BOTH w and c per h and c... precisely: reduce 'h w c -> h () c' with 'max' takes each row's per-channel max over width; then subtract that from the image (broadcasting back). Return img - mx. (uint8 wraparound on underflow is expected — same on both sides.)


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img = arr[0]

def solve(img):
    # Write your solution here
    return None

print(solve(img))


<!-- dd:dd-q340 -->

### Problem 340 · independent

Write a function solve(x) that takes a 4-D tensor of shape (b, c, h, w) and CENTERS each (batch, channel) feature map: subtract from every pixel the mean over ITS OWN map's h and w. Return the same shape; the trick is reducing to 'b c 1 1' so the means broadcast back.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x):
    """Implement the described contraction and return the result."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


<!-- dd:dd-q370 -->

### Problem 370 · independent

Write a function solve(x) that takes a batch (b, c, h, w) and centers each CHANNEL across the WHOLE BATCH and all pixels: subtract from every value its channel's mean over (b, h, w). Reduce to '1 c 1 1' so the means broadcast — BatchNorm-style centering (contrast per-image centering, which keeps b).


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x):
    """Implement the described contraction and return the result."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


#### Common mistakes

- **"reduce is rearrange with a mode argument."** — rearrange forbids
  dropping names; reduce requires it (something must reduce). They share
  the pattern grammar, not the contract.
- **"Means still need dividing outside, like einsum."** — reduce's third
  argument does real means/maxes/mins natively. The divide-outside habit
  is einsum-specific.
- **"'h w -> w' and 'h w -> 1 w' are the same reduction."** — Same numbers,
  different SHAPE: (w,) vs (1, w). The singleton version survives
  broadcasting against the original; the bare version may misalign
  (np-3's keepdims lesson, verbatim). Drills specify which they grade.


<!-- dd:dd-kp-einops-pooling -->

## Pooling with factored axes

`einops.pooling`


Combine reduce's aggregation with split's parentheses and you get POOLING —
window-wise downsampling — as pure notation:

> `einops.reduce(x, 'b c (h h2) (w w2) -> b c h w', 'mean', h2=2, w2=2)`

Read the input side as a split: height factors into (h blocks × h2 rows),
width into (w × w2). The output keeps the block coordinates (h, w) and
DROPS the within-window names (h2, w2) — so each output pixel aggregates
its h2×w2 window. That's non-overlapping average pooling; `'max'` makes it
max pooling. The mental model:

> **split the axis into (keep × window), reduce away the window.**

Variants the drills exercise:

- **Any window size / rectangle**: `h2=3, w2=3` for 3×3; the factors need
  not match.
- **Any rank**: a 5-D volume pools with three factored axes —
  `'b c (x a) (y b2) (z c2) -> b c x y z'` — nothing new, one more group.
- **Pool one axis only**: halve the width by averaging adjacent column
  pairs: `'b h (w w2) c -> b h w c', w2=2` — "adjacent pairs" is a
  length-2 window on that axis alone. Temporal downsampling
  ('b c (t two) -> b c t') is the same idea on sequences.
- **Pooling + flatten, etc.**: since it's all one pattern language, pooling
  composes freely with merges in the same call.

One requirement: non-overlapping windows must TILE the axis — sizes must
divide exactly (drills guarantee it; real code pads first). Overlapping /
strided pooling is outside reduce's power — that's `x.unfold(...)`
and the pooling layers' territory.


Task: 2×2 average pooling on a batch; then halving width by averaging
adjacent column pairs.


In [ ]:
import torch as t
import einops

x = t.arange(16.0).reshape(1, 1, 4, 4)      # (b, c, H, W)

# Split H into (2 blocks x 2 rows), W likewise; reduce the window names.
pooled = einops.reduce(x, 'b c (h h2) (w w2) -> b c h w', 'mean', h2=2, w2=2)
assert pooled.shape == (1, 1, 2, 2)
# Window (0,0) = mean of [[0,1],[4,5]] = 2.5:
assert pooled[0, 0].tolist() == [[2.5, 4.5],
                                 [10.5, 12.5]]

# Max pooling is the same pattern, different aggregation.
mx = einops.reduce(x, 'b c (h h2) (w w2) -> b c h w', 'max', h2=2, w2=2)
assert mx[0, 0].tolist() == [[5.0, 7.0],
                             [13.0, 15.0]]

# One-axis pooling: average adjacent COLUMN pairs, everything else intact.
imgs = t.arange(8.0).reshape(1, 2, 4, 1)    # (b, h, w=4, c)
halved = einops.reduce(imgs, 'b h (w w2) c -> b h w c', 'mean', w2=2)
assert halved.shape == (1, 2, 2, 1)
assert halved[0, 0, :, 0].tolist() == [0.5, 2.5]   # (0+1)/2, (2+3)/2
print(x[0, 0], "\n")
print("mean-pooled 2x2 ->\n", pooled[0, 0])
print("max-pooled  2x2 ->\n", mx[0, 0])
print("column pairs averaged:", imgs[0, 0, :, 0].tolist(), "->",
      halved[0, 0, :, 0].tolist())




Why each step:

1. Hand-verify ONE window ([[0,1],[4,5]] → 2.5) and trust the pattern for
   the rest — the same one-element discipline as every layout KP, now with
   an aggregation attached.
2. mean→max changing only the string argument shows where the operation
   lives: geometry in the pattern, semantics in the aggregation. Swap
   either independently.
3. In the one-axis case, note which name went INSIDE the parens: the axis
   being pooled. Everything outside parens rides along — that's how the
   pattern scales to 5-D volumes without new ideas.


In [ ]:
import torch as t
import einops

x = t.arange(16.0).reshape(1, 1, 4, 4)      # (b, c, H, W)

# Split H into (2 blocks x 2 rows), W likewise; reduce the window names.
pooled = einops.reduce(x, 'b c (h h2) (w w2) -> b c h w', 'mean', h2=2, w2=2)
assert pooled.shape == (1, 1, 2, 2)
# Window (0,0) = mean of [[0,1],[4,5]] = 2.5:
assert pooled[0, 0].tolist() == [[2.5, 4.5],
                                 [10.5, 12.5]]

# Max pooling is the same pattern, different aggregation.
mx = einops.reduce(x, 'b c (h h2) (w w2) -> b c h w', 'max', h2=2, w2=2)
assert mx[0, 0].tolist() == [[5.0, 7.0],
                             [13.0, 15.0]]

# One-axis pooling: average adjacent COLUMN pairs, everything else intact.
imgs = t.arange(8.0).reshape(1, 2, 4, 1)    # (b, h, w=4, c)
halved = einops.reduce(imgs, 'b h (w w2) c -> b h w c', 'mean', w2=2)
assert halved.shape == (1, 2, 2, 1)
assert halved[0, 0, :, 0].tolist() == [0.5, 2.5]   # (0+1)/2, (2+3)/2
print(x[0, 0], "\n")
print("mean-pooled 2x2 ->\n", pooled[0, 0])
print("max-pooled  2x2 ->\n", mx[0, 0])
print("column pairs averaged:", imgs[0, 0, :, 0].tolist(), "->",
      halved[0, 0, :, 0].tolist())


<!-- dd:dd-q324 -->

### Problem 324 · faded

2×2 non-overlapping average pooling on (B, C, H, W).


In [ ]:
import torch as t
import einops

def solve(x):
    """Halve H and W by averaging each 2x2 window."""
    return einops.reduce(x, '_____', 'mean', h2=2, w2=2)


<!-- dd:dd-q363 -->

### Problem 363 · guided

Write solve(img) for a float (C, H, W) image with H and W divisible by 3: 3×3 non-overlapping average pooling — H and W shrink by 3×, channels unchanged. einops.reduce with 'mean' and named window factors.


<details>
<summary>Hints</summary>

1. 3×3 average pooling on a channels-first single image (c, h, w) — same
   split-and-reduce, no batch axis.
2. Window factors are 3 this time; both spatial axes factor.
3. `'c (h h3) (w w3) -> c h w', h3=3, w3=3` with 'mean'.

</details>


In [ ]:
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy')).to(t.float32)
img = arr[0]

def solve(img):
    # Write your solution here
    return None

print(solve(img))


<!-- dd:dd-q386 -->

### Problem 386 · independent

Write solve(imgs) for a float (B, H, W, C) batch with W even: halve the WIDTH by averaging each pair of adjacent columns. Pattern: einops.reduce 'b h (w w2) c -> b h w c' with 'mean', w2=2.


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')).to(t.float32), 1, -1).flip(0)
imgs = arr

def solve(imgs):
    # Write your solution here
    return None

print(solve(imgs))


<!-- dd:dd-q368 -->

### Problem 368 · independent

Write a function solve(x3d) that takes a 5-D volume batch (b, c, x, y, z) with all three spatial dims EVEN and max-pools it with non-overlapping 2x2x2 windows: return shape (b, c, x//2, y//2, z//2) via 'b c (x 2) (y 2) (z 2) -> b c x y z' with 'max'. (Note the anonymous literal 2s in the pattern.)


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x3d):
    """Return shape (b, c, x//2, y//2, z//2) via 'b c (x 2) (y 2) (z 2) -> b c x y z' with 'max'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(1, 2, 2, 2, 2)))


<!-- dd:dd-q354 -->

### Problem 354 · independent

Write a function solve(arr) that takes a channels-last batch (b, h, w, c) and subtracts each (batch, channel) pair's SPATIAL MINIMUM from its own map: reduce with 'b h w c -> b () () c' and 'min', then subtract with broadcasting. Every map's minimum becomes exactly 0.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Implement the described contraction and return the result."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(4.0).reshape(1, 2, 2, 1)))


<!-- dd:dd-q336 -->

### Problem 336 · independent

Write a function solve(arr, k, b1) that takes a channels-first batch (b, c, h, w) — h, w divisible by k, b divisible by b1 — and does two things at once: max-pool each image with non-overlapping k x k windows AND tile the batch into a b1-row grid. Return shape (c, b1*(h//k), (b//b1)*(w//k)) via einops.reduce with pattern '(b1 b2) c (h h2) (w w2) -> c (b1 h) (b2 w)' and 'max'.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, k, b1):
    """Implement the described contraction and return the result."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(1, 1, 4, 4), 2, 1))


<!-- dd:dd-q377 -->

### Problem 377 · independent

Write a function solve(x, k) that takes a batch (b, c, h, w) — h, w divisible by k — and does max-pooling with k x k windows FOLLOWED by a full flatten: return shape (b, c*(h//k)*(w//k)) in ONE einops.reduce: 'b c (h k1) (w k2) -> b (c h w)' with 'max'.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, k):
    """Return shape (b, c*(h//k)*(w//k)) in ONE einops.reduce."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(1, 1, 4, 4), 2))


#### Common mistakes

- **"Pooling needs a framework (torch.nn.AvgPool2d) or a loop."** — Non-
  overlapping pooling is reshape+reduce, which is exactly what the factored
  pattern states. Frameworks add padding/stride options; the core is this.
- **"The window names (h2, w2) must be called that."** — Any names; what
  matters is they appear inside input parens and NOT in the output. The
  kept block-count names are the ones that survive.
- **"reduce can do stride-1 (overlapping) pooling too."** — No: factored
  axes tile the input disjointly. Overlap = sliding_window_view + reduction
  on `x.unfold(...)`. "Non-overlapping" in a task is your green light for
  the einops form.
